<a href="https://colab.research.google.com/github/Godswhill/Godswhill/blob/main/recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-surprise

In [2]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from nltk.stem.snowball import SnowballStemmer
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet
from surprise import Reader, Dataset, SVD

import warnings; warnings.simplefilter('ignore')

In [3]:
data = pd.read_csv("tmdb_10k_movies_full_data.csv")

In [4]:
data.head()

,id,imdb_id,adult,belongs_to_collection,budget,genres,homepage,original_language,original_title,overview,...,vote_average,vote_count,keywords,poster_path,cast,director,revenue (USD),production_companies,poster_url,trailer_url
0,574475,tt9619824,False,"{'id': 8864, 'name': 'Final Destination Collec...",50000000,"Horror, Mystery",https://www.finaldestinationmovie.com,en,Final Destination Bloodlines,"Plagued by a violent recurring nightmare, coll...",...,7.182,1357,"restaurant, gore, sequel, premonition, fate, f...",/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,"Kaitlyn Santa Juana, Teo Briones, Rya Kihlstedt",Adam B. Stein,281253961,"New Line Cinema, Practical Pictures, Freshman ...",https://image.tmdb.org/t/p/w500/6WxhEvFsauuACf...,https://www.youtube.com/watch?v=xitSoRbHJ50
1,552524,tt11655566,False,"{'id': 1504372, 'name': 'Lilo & Stitch (Live-A...",100000000,"Family, Science Fiction, Comedy, Adventure",https://movies.disney.com/lilo-and-stitch-2025,en,Lilo & Stitch,The wildly funny and touching story of a lonel...,...,7.095,733,"hawaii, bullying, dysfunctional family, alien,...",/7c5VBuCbjZOk7lSfj9sMpmDIaKX.jpg,"Maia Kealoha, Sydney Agudong, Chris Sanders",Dean Fleischer Camp,910349181,"Walt Disney Pictures, Rideback",https://image.tmdb.org/t/p/w500/7c5VBuCbjZOk7l...,https://www.youtube.com/watch?v=VWqJifMMgZE
2,1311844,tt31828378,False,NaN,0,"Action, Adventure, Drama",NaN,en,The Twisters,A deadly patchwork of destructive cyclones is ...,...,6.000,42,"tornado, twister, disaster movie, cyclone",/8OP3h80BzIDgmMNANVaYlQ6H4Oc.jpg,"Tiffany, Mark Justice, Kayla Fields",Michael Su,0,"Atomic Blonde Entertainment, The Global Asylum...",https://image.tmdb.org/t/p/w500/8OP3h80BzIDgmM...,https://www.youtube.com/watch?v=21_IhP-P9cU
3,605722,tt9883398,False,NaN,0,"Science Fiction, Comedy, Action",NaN,en,Distant,"After crash-landing on an alien planet, an ast...",...,6.300,142,"galaxy, alien, space, planet, spaceship, survi...",/czh8HOhsbBUKoKsmRmLQMCLHUev.jpg,"Anthony Ramos, Naomi Scott, Kristofer Hivju",Will Speck,0,"DreamWorks Pictures, Reliance Entertainment, A...",https://image.tmdb.org/t/p/w500/czh8HOhsbBUKoK...,https://www.youtube.com/watch?v=XBHNALPwxYE
4,1090007,tt26434746,False,NaN,0,"Crime, Thriller, Action",https://www.firstshiftmovie.com/,en,First Shift,NYPD veteran Mike and rookie Angela tackle a h...,...,5.391,23,NaN,/ajsGI4JYaciPIe3gPgiJ3Vw5Vre.jpg,"Gino Anthony Pesi, Kristen Renton, James McMen...",Uwe Boll,0,Event Filmproduktion,https://image.tmdb.org/t/p/w500/ajsGI4JYaciPIe...,https://www.youtube.com/watch?v=j9NVJ8Cbm-s


In [5]:
data.columns

Index(['id', 'imdb_id', 'adult', 'belongs_to_collection', 'budget', 'genres',
       'homepage', 'original_language', 'original_title', 'overview',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title', 'popularity', 'video', 'vote_average', 'vote_count',
       'keywords', 'poster_path', 'cast', 'director', 'revenue (USD)',
       'production_companies', 'poster_url', 'trailer_url'],
      dtype='object')

In [6]:
order = ['id', 'imdb_id', 'title', 'popularity', 'belongs_to_collection', 'budget', 'genres', 'adult',
       'homepage', 'original_language', 'original_title', 'overview',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'video', 'vote_average', 'vote_count',
       'keywords', 'poster_path', 'cast', 'director', 'revenue (USD)',
       'production_companies', 'poster_url', 'trailer_url']

In [7]:
data = data.reindex(columns=order)

In [8]:
data.head()

,id,imdb_id,title,popularity,belongs_to_collection,budget,genres,adult,homepage,original_language,...,vote_average,vote_count,keywords,poster_path,cast,director,revenue (USD),production_companies,poster_url,trailer_url
0,574475,tt9619824,Final Destination Bloodlines,629.4584,"{'id': 8864, 'name': 'Final Destination Collec...",50000000,"Horror, Mystery",False,https://www.finaldestinationmovie.com,en,...,7.182,1357,"restaurant, gore, sequel, premonition, fate, f...",/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,"Kaitlyn Santa Juana, Teo Briones, Rya Kihlstedt",Adam B. Stein,281253961,"New Line Cinema, Practical Pictures, Freshman ...",https://image.tmdb.org/t/p/w500/6WxhEvFsauuACf...,https://www.youtube.com/watch?v=xitSoRbHJ50
1,552524,tt11655566,Lilo & Stitch,400.0713,"{'id': 1504372, 'name': 'Lilo & Stitch (Live-A...",100000000,"Family, Science Fiction, Comedy, Adventure",False,https://movies.disney.com/lilo-and-stitch-2025,en,...,7.095,733,"hawaii, bullying, dysfunctional family, alien,...",/7c5VBuCbjZOk7lSfj9sMpmDIaKX.jpg,"Maia Kealoha, Sydney Agudong, Chris Sanders",Dean Fleischer Camp,910349181,"Walt Disney Pictures, Rideback",https://image.tmdb.org/t/p/w500/7c5VBuCbjZOk7l...,https://www.youtube.com/watch?v=VWqJifMMgZE
2,1311844,tt31828378,The Twisters,348.0535,NaN,0,"Action, Adventure, Drama",False,NaN,en,...,6.000,42,"tornado, twister, disaster movie, cyclone",/8OP3h80BzIDgmMNANVaYlQ6H4Oc.jpg,"Tiffany, Mark Justice, Kayla Fields",Michael Su,0,"Atomic Blonde Entertainment, The Global Asylum...",https://image.tmdb.org/t/p/w500/8OP3h80BzIDgmM...,https://www.youtube.com/watch?v=21_IhP-P9cU
3,605722,tt9883398,Distant,291.4148,NaN,0,"Science Fiction, Comedy, Action",False,NaN,en,...,6.300,142,"galaxy, alien, space, planet, spaceship, survi...",/czh8HOhsbBUKoKsmRmLQMCLHUev.jpg,"Anthony Ramos, Naomi Scott, Kristofer Hivju",Will Speck,0,"DreamWorks Pictures, Reliance Entertainment, A...",https://image.tmdb.org/t/p/w500/czh8HOhsbBUKoK...,https://www.youtube.com/watch?v=XBHNALPwxYE
4,1090007,tt26434746,First Shift,305.7355,NaN,0,"Crime, Thriller, Action",False,https://www.firstshiftmovie.com/,en,...,5.391,23,NaN,/ajsGI4JYaciPIe3gPgiJ3Vw5Vre.jpg,"Gino Anthony Pesi, Kristen Renton, James McMen...",Uwe Boll,0,Event Filmproduktion,https://image.tmdb.org/t/p/w500/ajsGI4JYaciPIe...,https://www.youtube.com/watch?v=j9NVJ8Cbm-s


In [9]:
data.duplicated().sum()

564

In [10]:
data.drop_duplicates(inplace=True)

In [11]:
data.duplicated().sum()

0

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9435 entries, 0 to 9998
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     9435 non-null   int64  
 1   imdb_id                9170 non-null   object 
 2   title                  9435 non-null   object 
 3   popularity             9435 non-null   float64
 4   belongs_to_collection  2589 non-null   object 
 5   budget                 9435 non-null   int64  
 6   genres                 9342 non-null   object 
 7   adult                  9435 non-null   bool   
 8   homepage               4062 non-null   object 
 9   original_language      9435 non-null   object 
 10  original_title         9435 non-null   object 
 11  overview               9369 non-null   object 
 12  release_date           9417 non-null   object 
 13  revenue                9435 non-null   int64  
 14  runtime                9435 non-null   int64  
 15  spoken_la

In [13]:
data.isnull().sum()

,0
id,0
imdb_id,265
title,0
popularity,0
belongs_to_collection,6846
budget,0
genres,93
adult,0
homepage,5373
original_language,0


In [14]:
data.belongs_to_collection.fillna("No Collection", inplace=True)

In [15]:
data.belongs_to_collection.head()

,belongs_to_collection
0,"{'id': 8864, 'name': 'Final Destination Collec..."
1,"{'id': 1504372, 'name': 'Lilo & Stitch (Live-A..."
2,No Collection
3,No Collection
4,No Collection


In [16]:
data.head()

,id,imdb_id,title,popularity,belongs_to_collection,budget,genres,adult,homepage,original_language,...,vote_average,vote_count,keywords,poster_path,cast,director,revenue (USD),production_companies,poster_url,trailer_url
0,574475,tt9619824,Final Destination Bloodlines,629.4584,"{'id': 8864, 'name': 'Final Destination Collec...",50000000,"Horror, Mystery",False,https://www.finaldestinationmovie.com,en,...,7.182,1357,"restaurant, gore, sequel, premonition, fate, f...",/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,"Kaitlyn Santa Juana, Teo Briones, Rya Kihlstedt",Adam B. Stein,281253961,"New Line Cinema, Practical Pictures, Freshman ...",https://image.tmdb.org/t/p/w500/6WxhEvFsauuACf...,https://www.youtube.com/watch?v=xitSoRbHJ50
1,552524,tt11655566,Lilo & Stitch,400.0713,"{'id': 1504372, 'name': 'Lilo & Stitch (Live-A...",100000000,"Family, Science Fiction, Comedy, Adventure",False,https://movies.disney.com/lilo-and-stitch-2025,en,...,7.095,733,"hawaii, bullying, dysfunctional family, alien,...",/7c5VBuCbjZOk7lSfj9sMpmDIaKX.jpg,"Maia Kealoha, Sydney Agudong, Chris Sanders",Dean Fleischer Camp,910349181,"Walt Disney Pictures, Rideback",https://image.tmdb.org/t/p/w500/7c5VBuCbjZOk7l...,https://www.youtube.com/watch?v=VWqJifMMgZE
2,1311844,tt31828378,The Twisters,348.0535,No Collection,0,"Action, Adventure, Drama",False,NaN,en,...,6.000,42,"tornado, twister, disaster movie, cyclone",/8OP3h80BzIDgmMNANVaYlQ6H4Oc.jpg,"Tiffany, Mark Justice, Kayla Fields",Michael Su,0,"Atomic Blonde Entertainment, The Global Asylum...",https://image.tmdb.org/t/p/w500/8OP3h80BzIDgmM...,https://www.youtube.com/watch?v=21_IhP-P9cU
3,605722,tt9883398,Distant,291.4148,No Collection,0,"Science Fiction, Comedy, Action",False,NaN,en,...,6.300,142,"galaxy, alien, space, planet, spaceship, survi...",/czh8HOhsbBUKoKsmRmLQMCLHUev.jpg,"Anthony Ramos, Naomi Scott, Kristofer Hivju",Will Speck,0,"DreamWorks Pictures, Reliance Entertainment, A...",https://image.tmdb.org/t/p/w500/czh8HOhsbBUKoK...,https://www.youtube.com/watch?v=XBHNALPwxYE
4,1090007,tt26434746,First Shift,305.7355,No Collection,0,"Crime, Thriller, Action",False,https://www.firstshiftmovie.com/,en,...,5.391,23,NaN,/ajsGI4JYaciPIe3gPgiJ3Vw5Vre.jpg,"Gino Anthony Pesi, Kristen Renton, James McMen...",Uwe Boll,0,Event Filmproduktion,https://image.tmdb.org/t/p/w500/ajsGI4JYaciPIe...,https://www.youtube.com/watch?v=j9NVJ8Cbm-s


In [18]:
def get_collection_name(collection_str):
    if isinstance(collection_str, str) and collection_str != 'No Collection':
        try:
            collection_dict = literal_eval(collection_str)
            return collection_dict.get('name')
        except (ValueError, SyntaxError):
            return None
    return None

data['collection_name'] = data['belongs_to_collection'].apply(get_collection_name)
display(data[['belongs_to_collection', 'collection_name']].head())

,belongs_to_collection,collection_name
0,"{'id': 8864, 'name': 'Final Destination Collec...",Final Destination Collection
1,"{'id': 1504372, 'name': 'Lilo & Stitch (Live-A...",Lilo & Stitch (Live-Action) Collection
2,No Collection,None
3,No Collection,None
4,No Collection,None


In [19]:
data


,id,imdb_id,title,popularity,belongs_to_collection,budget,genres,adult,homepage,original_language,...,vote_count,keywords,poster_path,cast,director,revenue (USD),production_companies,poster_url,trailer_url,collection_name
0,574475,tt9619824,Final Destination Bloodlines,629.4584,"{'id': 8864, 'name': 'Final Destination Collec...",50000000,"Horror, Mystery",False,https://www.finaldestinationmovie.com,en,...,1357,"restaurant, gore, sequel, premonition, fate, f...",/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,"Kaitlyn Santa Juana, Teo Briones, Rya Kihlstedt",Adam B. Stein,281253961,"New Line Cinema, Practical Pictures, Freshman ...",https://image.tmdb.org/t/p/w500/6WxhEvFsauuACf...,https://www.youtube.com/watch?v=xitSoRbHJ50,Final Destination Collection
1,552524,tt11655566,Lilo & Stitch,400.0713,"{'id': 1504372, 'name': 'Lilo & Stitch (Live-A...",100000000,"Family, Science Fiction, Comedy, Adventure",False,https://movies.disney.com/lilo-and-stitch-2025,en,...,733,"hawaii, bullying, dysfunctional family, alien,...",/7c5VBuCbjZOk7lSfj9sMpmDIaKX.jpg,"Maia Kealoha, Sydney Agudong, Chris Sanders",Dean Fleischer Camp,910349181,"Walt Disney Pictures, Rideback",https://image.tmdb.org/t/p/w500/7c5VBuCbjZOk7l...,https://www.youtube.com/watch?v=VWqJifMMgZE,Lilo & Stitch (Live-Action) Collection
2,1311844,tt31828378,The Twisters,348.0535,No Collection,0,"Action, Adventure, Drama",False,NaN,en,...,42,"tornado, twister, disaster movie, cyclone",/8OP3h80BzIDgmMNANVaYlQ6H4Oc.jpg,"Tiffany, Mark Justice, Kayla Fields",Michael Su,0,"Atomic Blonde Entertainment, The Global Asylum...",https://image.tmdb.org/t/p/w500/8OP3h80BzIDgmM...,https://www.youtube.com/watch?v=21_IhP-P9cU,None
3,605722,tt9883398,Distant,291.4148,No Collection,0,"Science Fiction, Comedy, Action",False,NaN,en,...,142,"galaxy, alien, space, planet, spaceship, survi...",/czh8HOhsbBUKoKsmRmLQMCLHUev.jpg,"Anthony Ramos, Naomi Scott, Kristofer Hivju",Will Speck,0,"DreamWorks Pictures, Reliance Entertainment, A...",https://image.tmdb.org/t/p/w500/czh8HOhsbBUKoK...,https://www.youtube.com/watch?v=XBHNALPwxYE,None
4,1090007,tt26434746,First Shift,305.7355,No Collection,0,"Crime, Thriller, Action",False,https://www.firstshiftmovie.com/,en,...,23,NaN,/ajsGI4JYaciPIe3gPgiJ3Vw5Vre.jpg,"Gino Anthony Pesi, Kristen Renton, James McMen...",Uwe Boll,0,Event Filmproduktion,https://image.tmdb.org/t/p/w500/ajsGI4JYaciPIe...,https://www.youtube.com/watch?v=j9NVJ8Cbm-s,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,347754,tt3212904,Curve,1.8606,No Collection,0,"Horror, Thriller",False,NaN,en,...,508,"hitchhiker, injured leg, car accident, trapped...",/9AGJMNrUfBMIGkCeKNccQbj0Nyg.jpg,"Julianne Hough, Teddy Sears, Madalyn Horcher",Iain Softley,0,"Universal Pictures, Blumhouse Productions, Est...",https://image.tmdb.org/t/p/w500/9AGJMNrUfBMIGk...,https://www.youtube.com/watch?v=n297-KgI5Sw,None
9995,80880,tt0431619,Aarya,1.9003,"{'id': 840690, 'name': 'Aarya Collection', 'po...",0,"Drama, Romance, Action",False,NaN,te,...,39,NaN,/1RU7eOgierTfTiNNQUmd1h5iKFU.jpg,"Allu Arjun, Anuradha Mehta, Siva Balaji",Sukumar,0,Sri Venkateswara Creations,https://image.tmdb.org/t/p/w500/1RU7eOgierTfTi...,NaN,Aarya Collection
9996,9421,tt0119038,The Dinner Game,2.1259,No Collection,12500000,Comedy,False,NaN,fr,...,1963,"tax inspector, fool",/jCOCdWIim6upzteYswh8BpcsKWg.jpg,"Jacques Villeret, Thierry Lhermitte, Francis H...",Francis Veber,78599508,"Gaumont, EFVE, TPS Cinéma, TF1 Films Production",https://image.tmdb.org/t/p/w500/jCOCdWIim6upzt...,https://www.youtube.com/watch?v=4FANGIUNbiA,None
9997,36200,tt0810831,Digimon Tamers: Battle of Adventurers,1.6942,"{'id': 1214254, 'name': 'Digimon Tamers Collec...",0,Animation,False,NaN,ja,...,16,"creature, anime, based on anime, proxy battle",/zZwRkkmhoGiMsrrCtxIuS9I7uvG.jpg,"Fumiko Orikasa, Makoto Tsumura, Mayumi Yamaguchi",Tetsuo Imazawa,0,Toei Animation,https://image.tmdb.org/t/p/w500/zZwRkkmhoGiMsr...,NaN,Digimon Tamers Collection


In [23]:
data[["title","collection_name","genres","release_date","popularity","vote_count","vote_average"]].sort_values("release_date",ascending=False).head(10)

,title,collection_name,genres,release_date,popularity,vote_count,vote_average
3110,Avengers: Secret Wars,The Avengers Collection,Science Fiction,2027-12-17,4.0577,0,0.0
327,How to Train Your Dragon 2,How to Train Your Dragon (Live-Action) Collection,"Fantasy, Adventure, Family, Action",2027-06-10,15.5455,0,0.0
5393,Spider-Man: Beyond the Spider-Verse,Spider-Man: Spider-Verse Collection,"Animation, Action, Adventure, Science Fiction",2027-06-02,3.5145,0,0.0
2762,Sonic the Hedgehog 4,Sonic the Hedgehog Collection,"Family, Comedy, Adventure, Science Fiction",2027-03-19,4.8582,0,0.0
3067,Shrek 5,Shrek Collection,"Animation, Family, Comedy, Fantasy",2026-12-23,4.4117,0,0.0
1204,Avengers: Doomsday,The Avengers Collection,Science Fiction,2026-12-18,8.5062,0,0.0
3399,Dune: Part Three,Dune Collection,"Adventure, Science Fiction",2026-12-16,3.4503,0,0.0
2190,Spider-Man: Brand New Day,Spider-Man (MCU) Collection,"Science Fiction, Action, Adventure",2026-07-29,5.2312,0,0.0
3940,The Odyssey,None,"Adventure, Drama, Fantasy",2026-07-15,7.7082,0,0.0
2467,Moana,None,"Adventure, Comedy, Family, Fantasy",2026-07-09,4.4476,0,0.0


In [24]:
data.status.unique()

array(['Released', 'Post Production', 'Planned', 'In Production',
       'Rumored', 'Canceled'], dtype=object)

In [51]:
(data[(data["status"]=="Released")
    & (data["vote_count"]>10)
    & (data["vote_average"]>0)])[["title","collection_name","genres","release_date","popularity","vote_count","vote_average"]] \
    .sort_values("release_date",ascending=False).head(10)

,title,collection_name,genres,release_date,popularity,vote_count,vote_average
22,Jurassic World Rebirth,Jurassic Park Collection,"Science Fiction, Adventure, Action",2025-07-01,116.8607,16,7.400
25,M3GAN 2.0,M3GAN Collection,"Action, Science Fiction",2025-06-25,103.0680,59,7.000
10,F1 The Movie,None,"Action, Drama",2025-06-23,233.6806,196,7.630
372,Trainwreck: Poop Cruise,Trainwreck,Documentary,2025-06-23,15.1513,22,5.400
236,Frozen: The Hit Broadway Musical,None,"Music, Family, Fantasy",2025-06-20,19.7305,18,7.167
993,Alma & the Wolf,None,"Horror, Mystery, Thriller",2025-06-20,7.8073,16,5.625
5,KPop Demon Hunters,None,"Animation, Fantasy, Action, Comedy, Music",2025-06-20,298.5944,268,8.612
247,Grenfell: Uncovered,None,Documentary,2025-06-19,18.1969,18,7.417
111,Semi-Soeter,Semi-Soet Collection,"Romance, Comedy",2025-06-19,29.1051,11,5.318
8,28 Years Later,28 Days/Weeks Later Collection,"Horror, Thriller, Science Fiction",2025-06-18,246.7163,363,7.100
